Install packages

In [ ]:
# ===== 1. SETUP (RUN THIS FIRST) =====
!apt-get update
!apt-get install -y sumo sumo-tools sumo-doc python3-tk
!pip install stable-baselines3==2.0.0 torch==2.0.1 traci sumolib pyvirtualdisplay gymnasium

# Setup environment
from pyvirtualdisplay import Display
display = Display(visible=0, size=(1024, 768))
display.start()

import os
import signal
import time
import numpy as np
import traci
import sumolib
from stable_baselines3 import PPO
import gymnasium as gym

# Cleanup function
def kill_sumo():
    os.system('pkill -f sumo')
    os.system('pkill -f sumo-gui')
    time.sleep(1)

# ===== 2. CREATE SIMULATION FILES =====
kill_sumo()

# Simple intersection
!netgenerate --grid --grid.number=2 --output-file=intersection.net.xml

# Traffic definition
with open("vehicles.rou.xml", "w") as f:
    f.write("""<routes>
    <vType id="car" accel="2.6" decel="4.5" length="5" maxSpeed="13.8"/>
    <route id="horizontal" edges="edge0 edge1"/>
    <route id="vertical" edges="edge2 edge3"/>
    <flow id="flow_h" route="horizontal" begin="0" end="3600" number="200" type="car"/>
    <flow id="flow_v" route="vertical" begin="0" end="3600" number="200" type="car"/>
</routes>""")

# Simulation config
with open("intersection.sumocfg", "w") as f:
    f.write("""<configuration>
    <input>
        <net-file value="intersection.net.xml"/>
        <route-files value="vehicles.rou.xml"/>
    </input>
    <time>
        <begin value="0"/>
        <end value="3600"/>
    </time>
</configuration>""")

# ===== 3. STABLE RL ENVIRONMENT =====
class TrafficLightEnv(gym.Env):
    def __init__(self):
        super().__init__()
        self.action_space = gym.spaces.Discrete(4)
        self.observation_space = gym.spaces.Box(low=0, high=100, shape=(4,))
        self.sumo_cmd = [
            sumolib.checkBinary('sumo'),
            "-c", "intersection.sumocfg",
            "--no-warnings",
            "--quit-on-end",
            "--no-step-log",
            "--random"
        ]
        self.current_step = 0
        self.max_steps = 3600

    def _get_state(self):
        try:
            return np.array([
                traci.lane.getLastStepHaltingNumber("edge0_0"),
                traci.lane.getLastStepHaltingNumber("edge1_0"),
                traci.lane.getLastStepHaltingNumber("edge2_0"),
                traci.lane.getLastStepHaltingNumber("edge3_0")
            ])
        except:
            return np.zeros(4)

    def reset(self, seed=None):
        kill_sumo()
        self.current_step = 0
        traci.start(self.sumo_cmd)
        return self._get_state(), {}

    def step(self, action):
        self.current_step += 1
        try:
            traci.trafficlight.setPhase("junction0", action)
            traci.simulationStep()
            state = self._get_state()
            reward = -np.sum(state)
            done = self.current_step >= self.max_steps
            return state, reward, done, False, {}
        except:
            return np.zeros(4), 0, True, False, {}

# ===== 4. TRAINING WITH ERROR RECOVERY =====
env = TrafficLightEnv()

model = PPO(
    "MlpPolicy",
    env,
    device="cpu",
    verbose=1,
    batch_size=16,  # Reduced for stability
    n_steps=64,
    learning_rate=2e-4,
    ent_coef=0.02
)

# Training loop with progress saving
for attempt in range(5):
    try:
        model.learn(
            total_timesteps=5000,  # Reduced for demo
            reset_num_timesteps=False,
            tb_log_name="ppo"
        )
        print(f"✅ Training completed (attempt {attempt+1})")
        break
    except Exception as e:
        print(f"⚠️ Attempt {attempt+1} failed: {str(e)}")
        kill_sumo()
        if attempt == 4:
            raise RuntimeError("Training failed after 5 attempts")
        time.sleep(2)

# ===== 5. EVALUATION =====
def run_simulation(controller):
    env = TrafficLightEnv()
    obs, _ = env.reset()
    total_wait = 0

    for _ in range(3600):
        if controller == "RL":
            action, _ = model.predict(obs, deterministic=True)
        else:  # Fixed-time
            action = int((traci.simulation.getTime() // 30) % 4)

        obs, _, terminated, truncated, _ = env.step(action)
        total_wait += -np.sum(obs)

        if terminated or truncated:
            break

    traci.close()
    return total_wait / 3600  # Average wait per second

print("\n🚦 Performance Results:")
print(f"AI Controller: {run_simulation('RL'):.2f} avg wait/sec")
print(f"Fixed-Time: {run_simulation('fixed'):.2f} avg wait/sec")

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading

RuntimeError: Training failed after 3 attempts

generate sumo files

Success.


define gym environment

train RL agent

SUMO path: /usr/share/sumo/bin/sumo
SUMO failed to start: Connection 'default' is already active.


 Retrying in 1 seconds


FatalTraCIError: Connection closed by SUMO.

evaluate performance

AttributeError: 'TrafficLightEnv' object has no attribute 'rewards'